In [12]:
import numpy as np
import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from tqdm import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr


rng = np.random.default_rng(seed=42)


N_SPECIES    = 101       # 1 pathogen + 50 specialists + 50 generalists
N_RESOURCES  = 20        # number of nutrient types 
N_GENES      = 8         # binary genome features per species

PATHOGEN_IDX = 0
N_COMMUNITY  = N_SPECIES - 1    # 100 non-pathogen species
N_SPECIALIST = N_COMMUNITY // 2 # 50
N_GENERALIST = N_COMMUNITY - N_SPECIALIST  # 50

T_MEASURE = 14.0

DIVERSITY_LEVELS = [1, 2, 4, 8, 16]
# These mirror the 5 resource-diversity levels used in the wet lab
# (1, 2, 4, 8, 16 carbon sources). We extend to 20 total resources
# so diversity 16 leaves 4 resources inactive.

N_REPLICATES_PER_LEVEL = {1: 10, 2: 8, 4: 6, 8: 4, 16: 4}
# 4 random resource combinations per diversity level = 20 broths total.
# Each replicate draws a different random subset of the active nutrients.

N_BROTHS = sum(N_REPLICATES_PER_LEVEL.values())   # = 20

TOTAL_CARBON_SUPPLY = 20.0

# This mirrors the wet-lab constraint: "final concentration 0.2% (w/v)
# maintained regardless of how many resources are in the mixture."
# Effect: 1 active nutrient gets R0=20.0, 16 active nutrients each get R0=1.25.

COMMUNITY_SIZES    = [12, 16, 24, 32, 48]

N_SAMPLES_PER_SIZE = 300
# 300 × 5 sizes × 20 broths = 30,000 unique communities.
# the more examples for XGBoost, the better

UPTAKE_SPARSITY = {
    "pathogen"   : (0.45, 0.70),   
    "specialist" : (0.10, 0.25),
    "generalist" : (0.40, 0.65),
}
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 6

RESOURCE_NAMES = [f"c{k+1}"             for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}"  for i in range(N_SPECIES)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Tabular_Models_Dataset"

In [13]:
# species traits

def build_species_traits():
    """
    Generates genome and uptake properties for all 101 species.

    Each species gets:
      genes[i]      : binary vector of length N_GENES (Bernoulli 0.5)
      c_matrix[i]   : uptake rates ciα in (0,1), computed as sigmoid of
                      (genes @ W + bias), then masked to zero outside active
                      nutrients. The sigmoid ensures all values are positive
                      and bounded.
      species_type  : 'pathogen', 'specialist', or 'generalist'

    Specialist vs generalist distinction:
      Specialists have narrow uptake (2–5 active nutrients, 10–25% of 20).
      Generalists have broad uptake (8–13 active nutrients, 40–65% of 20).
      Pathogen has broad uptake (9–14 nutrients, 45–70%) (more like generalist to make it a bit harder to suppress)
    """
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix     = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask  = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype = "pathogen";   pref = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"; pref = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"; pref = -1

        species_type.append(stype)
        preferred_nutrient.append(pref)

        lo, hi   = UPTAKE_SPARSITY[stype]
        n_active = max(1, int(round(rng.uniform(lo, hi) * N_RESOURCES)))

        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active_indices:
            active_indices[0] = pref
        uptake_mask[i, active_indices] = True

        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        raw = genes[i].astype(float) @ W + bias
        # sigmoid: 1 / (1 + exp(-raw)) maps any real number to (0,1)
        c_matrix[i] = (1.0 / (1.0 + np.exp(-raw))) * uptake_mask[i]

    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient

In [14]:
# broths

def build_broths():
    """
    Builds 20 broth environments structured by resource diversity.

    WET-LAB ALIGNMENT:
    The wet lab used 5 diversity levels (1,2,4,8,16 carbon sources) with the
    SAME total carbon concentration (0.2% w/v) in all conditions. 
    Mirroring this exactly: each active nutrient receives an equal share of
    TOTAL_CARBON_SUPPLY=20.0, with small random jitter (±15%) to avoid
    perfectly degenerate environments, then re-normalised to the exact total.

    For each diversity level we generate N_REPLICATES_PER_LEVEL=4 independent
    random resource combinations. Each replicate draws a different random subset
    of n active nutrients from the pool of 20.

    This creates a clean, interpretable structure:
      - All broths with the same diversity level have the same total carbon
      - Broths WITHIN a diversity level differ in WHICH nutrients are present
      - Broths ACROSS diversity levels differ in HOW MANY nutrients are present

    """
    broths = []
    broth_idx = 0

    for n_active in DIVERSITY_LEVELS:
        for rep in range(N_REPLICATES_PER_LEVEL[n_active]):

            # choosing active nutrients
            active_set = sorted(
                rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
            )

            R0 = np.zeros(N_RESOURCES)
            for k in active_set:
                R0[k] = (TOTAL_CARBON_SUPPLY / n_active) * rng.uniform(0.85, 1.15)
            # Renormalize to exact total
            if R0.sum() > 1e-12:
                R0 = R0 / R0.sum() * TOTAL_CARBON_SUPPLY
            # All inactive nutrients remain exactly 0 (no background supply).
            # This is the key difference from v6/v7b which had background
            # supply for all nutrients.

            # Derived features
            frac_R0     = R0 / (R0.sum() + 1e-12)
            active_flag = np.array([1 if k in active_set else 0
                                    for k in range(N_RESOURCES)], dtype=int)

            # Other broth parameters (unchanged from v7b)
            delta = float(rng.uniform(0.05, 0.20))
            w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
            l     = rng.uniform(0.1, 0.3,  size=N_RESOURCES)

            broth_name = f"Broth_{broth_idx+1:02d}"
            broths.append({
                "name"            : broth_name,
                "diversity_level" : n_active,
                "replicate"       : rep + 1,
                "active_set"      : active_set,
                "R0"              : R0,
                "frac_R0"         : frac_R0,
                "active_flag"     : active_flag,
                "w"               : w,
                "l"               : l,
                "delta"           : delta,
            })
            broth_idx += 1

    return broths

In [15]:
# crossfeed matrices

def build_species_crossfeed_matrices(species_type):
    """
    Each species i gets its own D_i matrix (N_RESOURCES × N_RESOURCES).

    D_i[alpha, beta] = fraction of the leaked metabolic flux from species i
    consuming nutrient beta that becomes nutrient alpha.

    Each column of D_i sums to 1 (Dirichlet sampling ensures flux conservation).
    The number of active output slots per column is drawn from [2, 6].

    Species-type-specific biases:
      Specialists: biased toward their preferred nutrient as output
      Pathogen:    uniform weights
      Generalists: moderate, slightly more uniform weights
    """
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            n_out        = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.array([
                    rng.uniform(2.0, 5.0) if a == pref and beta != pref
                    else rng.uniform(0.5, 2.0)
                    for a in active_alpha
                ])
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            probs = rng.dirichlet(alpha_vec)
            D_i[active_alpha, beta] = probs
        D_list.append(D_i)
    return D_list

In [16]:
# ode simulation

def micrm_ode(t, y, c, w, l, m, g_rate, delta, R0, D_list_sub):
    """
    Right-hand side of the MiCRM ODE system.

    State vector y = [N_0, N_1, ..., N_{n_sp-1}, R_0, R_1, ..., R_{n_res-1}]

    Species dynamics:
      dNi/dt = Ni * [gi * sum_alpha(w_alpha * (1-l_alpha) * c_i_alpha * R_alpha) - mi]
               - delta * Ni
    The delta*Ni term is dilution (serial passage in wet lab).

    Resource dynamics:
      dRalpha/dt = delta*(R0_alpha - R_alpha)    ← dilution toward initial supply
                 - sum_i(c_i_alpha * Ni * R_alpha) ← consumption by all species
                 + crossfeed_alpha               ← metabolic byproduct input
    """
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g_rate * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        Ni = N[i]
        if Ni < 1e-14:
            continue
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            flux_i_beta = l[beta] * c[i, beta] * Ni * R[beta]
            if flux_i_beta < 1e-16:
                continue
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]
            for alpha in nz:
                crossfeed[alpha] += (col[alpha] * w[beta] /
                                     (w[alpha] + 1e-12) * flux_i_beta)

    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=T_MEASURE):
    """
    Run MiCRM for a given subset of species in a given broth.

    species_subset : list of global species indices to include
    Returns: N_out (abundances at T_MEASURE), R_final, sol object
    """
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]

    y0  = np.concatenate([np.full(len(species_subset), 0.01), R0.copy()])
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, max(50, int(t_end * 5))),
        rtol   = 1e-5,
        atol   = 1e-7,
    )
    y_final = sol.y[:, -1]
    N_out   = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = max(0.0, y_final[local_idx])
    R_final = np.maximum(y_final[len(species_subset):], 0.0)
    return N_out, R_final, sol

In [17]:
# suppression calculations

def calc_suppression(N_path_measured, N_path_solo):
    """
    Community suppression score in [0, 1].
    = 1 - (pathogen abundance with community) / (pathogen abundance alone)
    0 = no suppression, 1 = full suppression.
    """
    eps = 1e-12
    if N_path_solo < eps:
        return 0.0
    return float(np.clip(1.0 - N_path_measured / (N_path_solo + eps), 0.0, 1.0))


def _lum_transform(supp_score, N_path_solo):
    """
    Log-normalised luminescence proxy:
    ss_lum = log10(1 - suppression) ∈ [-10, 0]
    Mirrors real plate-reader luminescence normalisation in the wet lab.
    """
    ratio = max(1.0 - supp_score, 1e-10)
    return round(float(np.clip(np.log10(ratio), -10.0, 0.0)), 6)

In [18]:
# =============================================================================
# 6.  MAIN DATASET GENERATION
# =============================================================================
def generate_dataset():
    print("=" * 65)
    print("  MiCRM v9 — Nutrient-Diversity-Aware Dataset")
    print(f"  T_MEASURE = {T_MEASURE}")
    print(f"  Diversity levels: {DIVERSITY_LEVELS} (carbon sources)")
    print(f"  Replicates per level: {N_REPLICATES_PER_LEVEL}  →  {N_BROTHS} broths total")
    print(f"  Community sizes: {COMMUNITY_SIZES}")
    print(f"  Samples per size per broth: {N_SAMPLES_PER_SIZE}")
    total_comm = N_BROTHS * len(COMMUNITY_SIZES) * N_SAMPLES_PER_SIZE
    print(f"  Expected communities: {total_comm:,}")
    print(f"  Total carbon supply (constant): {TOTAL_CARBON_SUPPLY}")
    print("=" * 65)

    # ── Build shared ecology ───────────────────────────────────────────────────
    print("\n[1] Building species traits ...")
    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()

    print("[2] Building broth environments (diversity-structured, v8) ...")
    broths = build_broths()
    for b in broths:
        active_names = [f"c{k+1}" for k in b["active_set"]]
        print(f"   {b['name']:10s}  diversity={b['diversity_level']:>2}  "
              f"active={active_names}  R0_sum={b['R0'].sum():.2f}")

    print("[3] Building cross-feeding matrices ...")
    D_list = build_species_crossfeed_matrices(species_type)

    # Maintenance cost and growth rate
    # v7b: pathogen has LOWER maintenance to make it harder to suppress
    m = rng.uniform(0.1, 0.5, size=N_SPECIES)
    g = rng.uniform(0.5, 1.5, size=N_SPECIES)
    m[PATHOGEN_IDX] = float(rng.uniform(0.05, 0.12))

    # ── Step 4: Pathogen solo abundances ──────────────────────────────────────
    # Run pathogen alone in each of the 20 broths.
    # This gives the denominator for all suppression calculations.
    print(f"\n[4] Pathogen solo abundances (across all {N_BROTHS} broths) ...")
    pathogen_solo = {}
    for broth in tqdm(broths, desc="  Pathogen solo", unit="broth"):
        N_s, _, _ = run_micrm(c_matrix, broth["w"], broth["l"], m, g,
                               broth["delta"], broth["R0"], D_list,
                               species_subset=[PATHOGEN_IDX])
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])

    # ── Step 5: Monoculture (single-strain) suppression ───────────────────────
    # For each of the 100 non-pathogen strains in each of the 20 broths:
    # run strain + pathogen together, measure pathogen abundance.
    # This gives ss_suppression_raw = individual suppression ability.
    # WET-LAB ALIGNMENT: "measure how each strain grew in isolation in each environment"
    # We run strain + pathogen (pairwise), not truly single-strain, because we need
    # the suppression effect on the pathogen. True monoculture of each strain is
    # also computed in build_monoculture_dataset() for reference.
    print(f"\n[5] Single-strain suppression (ss_lum, 100 strains × {N_BROTHS} broths) ...")
    ss_lum = {}
    strain_indices = list(range(1, N_SPECIES))
    for broth in tqdm(broths, desc="  Pairwise ODE", unit="broth"):
        bname    = broth["name"]
        N_solo_p = pathogen_solo[bname]
        ss_lum[bname] = {}
        for sp_idx in strain_indices:
            N_pair, _, _ = run_micrm(c_matrix, broth["w"], broth["l"], m, g,
                                      broth["delta"], broth["R0"], D_list,
                                      species_subset=[sp_idx, PATHOGEN_IDX])
            ss_lum[bname][sp_idx] = calc_suppression(N_pair[PATHOGEN_IDX], N_solo_p)

    # ── Step 6: Community suppression ─────────────────────────────────────────
    # For each (broth × community_size × sample), draw a random community of
    # N non-pathogen strains, run with pathogen, measure suppression.
    print(f"\n[6] Community suppression ({total_comm:,} communities) ...")
    community_rows   = []
    community_id_ctr = 0

    for broth in tqdm(broths, desc="  Community ODE", unit="broth"):
        bname      = broth["name"]
        N_solo_p   = pathogen_solo[bname]
        dlevel     = broth["diversity_level"]

        for comm_size in COMMUNITY_SIZES:
            for _ in range(N_SAMPLES_PER_SIZE):
                # Random community — uniform sampling without replacement
                members = sorted(
                    rng.choice(strain_indices, size=comm_size, replace=False).tolist()
                )
                N_comm, _, _ = run_micrm(
                    c_matrix, broth["w"], broth["l"], m, g,
                    broth["delta"], broth["R0"], D_list,
                    species_subset=members + [PATHOGEN_IDX]
                )
                comm_supp = calc_suppression(N_comm[PATHOGEN_IDX], N_solo_p)
                comm_id   = f"Comm_{community_id_ctr:06d}"
                community_id_ctr += 1

                for sp_idx in members:
                    row = {
                        # ── Community metadata ──────────────────────────────
                        "community_id"           : comm_id,
                        "broth_id"               : bname,
                        "diversity_level"        : dlevel,
                        # NEW v8: which diversity level this broth belongs to
                        "community_size"         : comm_size,
                        "community_suppression"  : round(comm_supp, 6),

                        # ── Broth metadata ───────────────────────────────────
                        "delta_per_h"            : round(broth["delta"], 4),
                        "R0_sum"                 : round(float(broth["R0"].sum()), 4),
                        # R0_sum is constant = TOTAL_CARBON_SUPPLY for all broths

                        # ── Strain identity ──────────────────────────────────
                        "strain_id"              : SPECIES_NAMES[sp_idx],
                        "species_type"           : species_type[sp_idx],
                        "maintenance_cost"       : round(float(m[sp_idx]), 4),
                        "growth_rate"            : round(float(g[sp_idx]), 4),

                        # ── Individual suppression scores ────────────────────
                        # These are NOT used as XGBoost node features (the goal
                        # is to predict from composition + nutrients only), but
                        # are recorded for validation and comparison purposes.
                        "ss_luminescence"        : _lum_transform(
                            ss_lum[bname][sp_idx], pathogen_solo[bname]
                        ),
                        "ss_suppression_raw"     : round(ss_lum[bname][sp_idx], 6),
                    }

                    # ── Genes (binary genome) ─────────────────────────────────
                    for g_idx in range(N_GENES):
                        row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])

                    # ── Uptake rates ──────────────────────────────────────────
                    for k in range(N_RESOURCES):
                        row[f"uptake_c{k+1}"] = round(float(c_matrix[sp_idx, k]), 6)

                    # ── Nutrient features — KEY XGBoost inputs ────────────────
                    # R0_c1..c20: raw initial concentrations
                    # frac_c1..c20: fractional supply (= R0 / sum(R0))
                    # active_c1..c20: binary flag (1 = this nutrient is supplied)
                    for k in range(N_RESOURCES):
                        row[f"R0_c{k+1}"]     = round(float(broth["R0"][k]),       4)
                        row[f"frac_c{k+1}"]   = round(float(broth["frac_R0"][k]),  6)
                        row[f"active_c{k+1}"] = int(broth["active_flag"][k])

                    community_rows.append(row)

    df = pd.DataFrame(community_rows)
    print(f"\n  Total rows (strain × community): {len(df):,}")
    print(f"  Unique communities             : {df['community_id'].nunique():,}")
    print(f"\n  community_suppression summary:")
    print(df["community_suppression"].describe().round(4))
    print(f"\n  By diversity level:")
    print(df.groupby("diversity_level")["community_suppression"]
          .agg(["mean","std","count"]).round(3))

    return df, genes, c_matrix, uptake_mask, m, g, broths, D_list, species_type, ss_lum, pathogen_solo

In [19]:
# =============================================================================
# 7.  COMMUNITY INDEX  (XGBoost-ready tabular format)
# =============================================================================
def build_community_index(df):
    """
    Builds the XGBoost-ready community index.

    One row per community. Contains:
      - community_id, broth_id, diversity_level, community_size
      - community_suppression (TARGET)
      - R0_c1..R0_c20 (20 nutrient features)
      - active_c1..active_c20 (20 binary nutrient flags)
      - Species_0002..Species_0101 (100 binary membership columns)

    TOTAL XGBoost features: 20 (R0) + 20 (active) + 100 (membership) = 140
    Plus community_size and diversity_level = 142 features for prediction.

    The binary membership vector is the core innovation: it encodes WHO is
    in the community as a 100-dimensional binary vector. Combined with
    the nutrient features, this gives XGBoost the full information needed.
    """
    # Base index: one row per community
    idx = df.groupby("community_id").agg(
        broth_id              = ("broth_id",              "first"),
        diversity_level       = ("diversity_level",       "first"),
        community_size        = ("community_size",        "first"),
        community_suppression = ("community_suppression", "first"),
        delta_per_h           = ("delta_per_h",           "first"),
        R0_sum                = ("R0_sum",                "first"),
        n_members             = ("strain_id",             "count"),
    ).reset_index()

    # ── Add nutrient features (one row per community, same for all members) ───
    r0_cols     = [f"R0_c{k+1}"     for k in range(N_RESOURCES)]
    frac_cols   = [f"frac_c{k+1}"   for k in range(N_RESOURCES)]
    active_cols = [f"active_c{k+1}" for k in range(N_RESOURCES)]
    nutrient_cols = r0_cols + frac_cols + active_cols

    nutrient_df = df.groupby("community_id")[nutrient_cols].first().reset_index()
    idx = idx.merge(nutrient_df, on="community_id")

    # ── Add binary membership matrix (100 species columns) ───────────────────
    # For each community, which of the 100 species are present?
    # membership_df: rows=communities, columns=species, values=0/1
    membership_df = (
        df[["community_id", "strain_id"]]
        .assign(present=1)
        .pivot_table(
            index   = "community_id",
            columns = "strain_id",
            values  = "present",
            fill_value = 0,
            aggfunc = "max",
        )
        .reset_index()
    )
    # Sort species columns for consistency
    species_cols = sorted([c for c in membership_df.columns if c.startswith("Species_")])
    membership_df = membership_df[["community_id"] + species_cols]

    idx = idx.merge(membership_df, on="community_id")

    print(f"  Community index: {len(idx):,} rows × {len(idx.columns)} cols")
    print(f"  XGBoost feature count: 142 (R0×20 + active×20 + membership×100 + size + level)")
    return idx


# =============================================================================
# 8.  MONOCULTURE DATASET  (reference — not XGBoost input features)
# =============================================================================
def build_monoculture_dataset(genes, c_matrix, m, g, broths, species_type, ss_lum, pathogen_solo):
    """
    One row per strain. Records:
      - Genomic features (genes, uptake rates) for reference
      - ss_suppression_raw averaged across all 20 broths
      - ss_suppression per diversity level (average across replicates)
    WET-LAB ALIGNMENT: corresponds to measuring each strain in isolation in
    each environment (monoculture + sentinel strain assay).
    NOTE: these individual scores are NOT input features for XGBoost — they
    would constitute data leakage if used alongside community composition.
    """
    print("\n[7] Building monoculture dataset ...")
    ref_broth = broths[0]

    # Suppression averaged by diversity level
    level_supp = {lvl: [] for lvl in DIVERSITY_LEVELS}
    for b in broths:
        lvl = b["diversity_level"]
        for sp_idx in range(1, N_SPECIES):
            level_supp[lvl].append(ss_lum[b["name"]][sp_idx])

    mono_rows = []
    for sp_idx in range(1, N_SPECIES):
        all_supp = [ss_lum[b["name"]][sp_idx] for b in broths]
        row = {
            "strain_id"        : SPECIES_NAMES[sp_idx],
            "species_type"     : species_type[sp_idx],
            "maintenance_cost" : round(float(m[sp_idx]), 4),
            "growth_rate"      : round(float(g[sp_idx]), 4),
            "ss_luminescence"  : _lum_transform(
                float(np.mean(all_supp)),
                np.mean([pathogen_solo[b["name"]] for b in broths])
            ),
            "ss_suppression_mean" : round(float(np.mean(all_supp)), 6),
        }
        # Per-diversity-level mean suppression
        for lvl in DIVERSITY_LEVELS:
            supp_at_lvl = [ss_lum[b["name"]][sp_idx]
                           for b in broths if b["diversity_level"] == lvl]
            row[f"ss_supp_d{lvl}"] = round(float(np.mean(supp_at_lvl)), 6)

        for g_idx in range(N_GENES):
            row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
        for k in range(N_RESOURCES):
            row[f"uptake_c{k+1}"] = round(float(c_matrix[sp_idx, k]), 6)
        # R0 from reference broth (diversity level 1) for reference
        for k in range(N_RESOURCES):
            row[f"R0_c{k+1}"] = round(float(ref_broth["R0"][k]), 4)
        mono_rows.append(row)

    df_mono = pd.DataFrame(mono_rows)
    print(f"  Monoculture dataset: {len(df_mono)} rows × {len(df_mono.columns)} cols")
    return df_mono

In [20]:
# =============================================================================
# 9.  FIGURES
# =============================================================================
def make_figures(df, df_idx, df_mono, broths, pathogen_solo, OUTPUT_DIR):
    """
    Generates 11 diagnostic figures matching the v6 MiCRM report structure,
    updated to reflect the v8 diversity-level design.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    TEAL   = "#1a6b6b"
    ORANGE = "#d4720c"
    NAVY   = "#1c2e4a"
    GREY   = "#78909C"
    palette5 = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]
    # 5 colours for 5 diversity levels

    r0_cols = [f"R0_c{k+1}" for k in range(N_RESOURCES)]

    plt.rcParams.update({
        "figure.facecolor" : "white",
        "axes.facecolor"   : "white",
        "axes.spines.top"  : False,
        "axes.spines.right": False,
        "font.family"      : "sans-serif",
        "font.size"        : 11,
    })

    # ── Figure 1: Dataset Quality Check ───────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Figure 1 — Dataset Quality Check  (v9, T_MEASURE={T_MEASURE})",
                 fontsize=13, fontweight="bold")

    # Left: community suppression distribution
    axes[0].hist(df_idx["community_suppression"], bins=40,
                 color=TEAL, edgecolor="white", alpha=0.85)
    frac_hi = (df_idx["community_suppression"] > 0.95).mean()
    axes[0].axvline(0.95, color="red", ls="--", lw=1.5,
                    label=f"{frac_hi:.1%} fully suppressed")
    axes[0].set_title("Community Suppression Distribution\n(good: spread across [0,1])")
    axes[0].set_xlabel("Community suppression")
    axes[0].set_ylabel("Count")
    axes[0].legend(fontsize=9)

    # Middle: ss_suppression_raw distribution
    axes[1].hist(df["ss_suppression_raw"], bins=40,
                 color=ORANGE, edgecolor="white", alpha=0.85)
    axes[1].axvspan(0, 1, alpha=0.08, color="green",
                    label="Valid range [0, 1]")
    axes[1].set_title("ss_suppression_raw Distribution\n(individual suppression ability)")
    axes[1].set_xlabel("ss_suppression_raw")
    axes[1].set_ylabel("Count")
    axes[1].legend(fontsize=9)

    # Right: pathogen solo abundance per diversity level (mean across replicates)
    levels = DIVERSITY_LEVELS
    mean_path_solo = []
    for lvl in levels:
        broth_names = [b["name"] for b in broths if b["diversity_level"] == lvl]
        mean_path_solo.append(np.mean([pathogen_solo[n] for n in broth_names]))
    axes[2].bar([str(l) for l in levels], mean_path_solo,
                color=TEAL, edgecolor="white", alpha=0.85)
    axes[2].set_title(f"Pathogen Solo N at T={T_MEASURE}\n(per diversity level, mean over replicates)")
    axes[2].set_xlabel("Diversity level (# active nutrients)")
    axes[2].set_ylabel("N_pathogen_solo")
    axes[2].set_xticks(range(len(levels)))
    axes[2].set_xticklabels([str(l) for l in levels])

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig1_dataset_quality_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig1_dataset_quality_v9.png")

    # ── Figure 2: Suppression vs Community Size ─────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 2 — Community Suppression vs Size",
                 fontsize=13, fontweight="bold")

    idx_plot = df_idx.copy()
    # Left: box plots by size
    idx_plot.boxplot("community_suppression", by="community_size",
                     ax=axes[0], boxprops=dict(color=TEAL),
                     medianprops=dict(color=ORANGE, lw=2),
                     whiskerprops=dict(color=TEAL),
                     capprops=dict(color=TEAL),
                     flierprops=dict(marker="o", ms=2, alpha=0.2, color=TEAL))
    axes[0].set_title("Suppression Distribution by Size")
    axes[0].set_xlabel("Community size N")
    axes[0].set_ylabel("Community suppression")
    plt.sca(axes[0])
    plt.title("Suppression Distribution by Size")

    # Middle: mean ± std by size
    grp = idx_plot.groupby("community_size")["community_suppression"]
    means = grp.mean(); stds = grp.std()
    sizes = means.index.tolist()
    axes[1].plot(range(len(sizes)), means.values, color=TEAL,
                 lw=2.5, marker="o", ms=7, markerfacecolor="white",
                 markeredgewidth=2, markeredgecolor=TEAL)
    axes[1].fill_between(range(len(sizes)),
                         means.values - stds.values,
                         means.values + stds.values,
                         alpha=0.15, color=TEAL)
    axes[1].set_xticks(range(len(sizes)))
    axes[1].set_xticklabels([str(s) for s in sizes])
    axes[1].set_title("Mean ± 1 std Suppression vs Size")
    axes[1].set_xlabel("Community size N")
    axes[1].set_ylabel("Community suppression")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    # Right: mean suppression by size × diversity level (heatmap)
    pivot = idx_plot.pivot_table(
        index="community_size", columns="diversity_level",
        values="community_suppression", aggfunc="mean")
    sns.heatmap(pivot, ax=axes[2], cmap="YlOrRd",
                annot=True, fmt=".2f", linewidths=0.5,
                cbar_kws={"label": "Mean suppression"})
    axes[2].set_title("Mean Suppression: Size × Diversity Level")
    axes[2].set_xlabel("Diversity level (# active nutrients)")
    axes[2].set_ylabel("Community size N")

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig2_suppression_by_size_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig2_suppression_by_size_v9.png")

    # ── Figure 3: Suppression vs Resource Diversity Level ───────────────────
    # KEY NEW FIGURE for v8: shows the core hypothesis
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 3 — Community Suppression vs Resource Diversity Level\n"
                 "(core hypothesis: more diverse nutrients → stronger competition → more suppression?)",
                 fontsize=12, fontweight="bold")

    level_col = [str(l) for l in DIVERSITY_LEVELS]
    # Left: box plots by diversity level
    idx_plot.boxplot("community_suppression", by="diversity_level",
                     ax=axes[0], boxprops=dict(color=TEAL),
                     medianprops=dict(color=ORANGE, lw=2),
                     whiskerprops=dict(color=TEAL),
                     capprops=dict(color=TEAL),
                     flierprops=dict(marker="o", ms=2, alpha=0.2, color=TEAL))
    plt.sca(axes[0])
    plt.title("Suppression by Diversity Level")
    axes[0].set_xlabel("# Active nutrients")
    axes[0].set_ylabel("Community suppression")

    # Middle: mean ± std by diversity level
    grp2 = idx_plot.groupby("diversity_level")["community_suppression"]
    means2 = grp2.mean(); stds2 = grp2.std()
    levels2 = means2.index.tolist()
    x_pos = range(len(levels2))
    axes[1].plot(x_pos, means2.values, color=TEAL, lw=2.5,
                 marker="s", ms=8, markerfacecolor="white",
                 markeredgewidth=2, markeredgecolor=TEAL)
    axes[1].fill_between(x_pos, means2.values - stds2.values,
                         means2.values + stds2.values, alpha=0.15, color=TEAL)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels([str(l) for l in levels2])
    axes[1].set_title("Mean ± 1 std Suppression vs Diversity Level")
    axes[1].set_xlabel("Diversity level (# active nutrients)")
    axes[1].set_ylabel("Community suppression")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    # Right: mean suppression by diversity level × community size
    pivot2 = idx_plot.pivot_table(
        index="diversity_level", columns="community_size",
        values="community_suppression", aggfunc="mean")
    for j, (sz, col) in enumerate(zip(pivot2.columns,
                                      ["#2196F3","#4CAF50","#FF9800","#E91E63","#9C27B0"])):
        axes[2].plot(x_pos, pivot2[sz].values, color=col, lw=2,
                     marker="o", ms=6, markerfacecolor="white",
                     markeredgewidth=1.5, label=f"N={sz}")
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels([str(l) for l in levels2])
    axes[2].set_title("Mean Suppression by Diversity Level × Size")
    axes[2].set_xlabel("Diversity level (# active nutrients)")
    axes[2].set_ylabel("Mean community suppression")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig3_suppression_vs_diversity_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig3_suppression_vs_diversity_v9.png")

    # ── Figure 4: Uptake Rate Sparsity ─────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 4 — Uptake Rate Sparsity (MiCRM Trait Structure)",
                 fontsize=13, fontweight="bold")

    # Left: heatmap of c_matrix (the full uptake matrix is stored in df_mono)
    uptake_col = [f"uptake_c{k+1}" for k in range(N_RESOURCES)]
    mat = df_mono[uptake_col].values
    im = axes[0].imshow(mat, aspect="auto", cmap="YlOrRd")
    axes[0].set_title("Uptake Rate Matrix ciα\n(white = masked/unavailable to species)")
    axes[0].set_xlabel("Nutrient α")
    axes[0].set_ylabel("Species i")
    plt.colorbar(im, ax=axes[0], label="Uptake rate ciα")

    # Middle: breadth distribution by species type
    df_mono_plot = df_mono.copy()
    df_mono_plot["n_active"] = (df_mono_plot[uptake_col] > 0).sum(axis=1)
    for stype, color, label in [("specialist", "#2196F3", "Specialist (n=50)"),
                                  ("generalist", "#4CAF50", "Generalist (n=50)")]:
        mask = df_mono_plot["species_type"] == stype
        axes[1].hist(df_mono_plot.loc[mask, "n_active"], bins=range(1, N_RESOURCES + 2),
                     alpha=0.7, color=color, edgecolor="white", label=label)
    axes[1].set_title("Uptake Breadth Distribution by Species Type")
    axes[1].set_xlabel("Number of active nutrients per species")
    axes[1].set_ylabel("Count")
    axes[1].legend(fontsize=9)

    # Right: gene presence frequency by type
    gene_cols = [f"gene_{k+1}" for k in range(N_GENES)]
    x_genes = range(N_GENES)
    for stype, color, label in [("specialist", "#2196F3", "Specialist"),
                                  ("generalist", "#4CAF50", "Generalist")]:
        mask = df_mono_plot["species_type"] == stype
        freq = df_mono_plot.loc[mask, gene_cols].mean().values
        axes[2].bar([x + (0.2 if stype == "generalist" else -0.2) for x in x_genes],
                    freq, width=0.35, alpha=0.8, color=color, label=label)
    axes[2].set_title("Gene Presence Frequency by Species Type")
    axes[2].set_xlabel("Gene")
    axes[2].set_ylabel("Fraction present (=1)")
    axes[2].set_xticks(range(N_GENES))
    axes[2].set_xticklabels([f"g{k+1}" for k in range(N_GENES)])
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig4_uptake_sparsity_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig4_uptake_sparsity_v9.png")

    # ── Figure 5: Community Membership Structure ──────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 5 — Community Membership Structure\n"
                 "(one community per row, ordered by size then broth)",
                 fontsize=12, fontweight="bold")

    # Left: binary membership matrix (80 random communities)
    species_cols = [c for c in df_idx.columns if c.startswith("Species_")]
    sample_ids = df_idx.groupby("community_size").apply(
        lambda g: g.sample(min(16, len(g)), random_state=42)
    )["community_id"].values
    sample_df = df_idx.set_index("community_id").loc[sample_ids, species_cols]
    axes[0].imshow(sample_df.values, aspect="auto", cmap="Blues",
                   interpolation="nearest", vmin=0, vmax=1)
    axes[0].set_title(f"Community × Strain Membership\n(blue = present; 80 sampled)")
    axes[0].set_xlabel("Strain (1–100)")
    axes[0].set_ylabel("Community")

    # Middle: communities per (diversity level × size)
    counts = df_idx.groupby(["diversity_level", "community_size"]).size().reset_index(name="n")
    pivot_m = counts.pivot(index="community_size", columns="diversity_level", values="n").fillna(0)
    bottom = np.zeros(len(pivot_m))
    for j, (col, color) in enumerate(zip(pivot_m.columns, palette5)):
        axes[1].bar(range(len(pivot_m)), pivot_m[col].values,
                    bottom=bottom, color=color, edgecolor="white",
                    label=f"D={col}")
        bottom += pivot_m[col].values
    axes[1].set_xticks(range(len(pivot_m)))
    axes[1].set_xticklabels([str(s) for s in pivot_m.index])
    axes[1].set_title("Communities per Size Bin\n(stacked by diversity level)")
    axes[1].set_xlabel("Community size N")
    axes[1].set_ylabel("Number of communities")
    axes[1].legend(fontsize=8, title="Div. level")

    # Right: strain sampling frequency
    strain_freq = df_idx[species_cols].sum()
    axes[2].hist(strain_freq.values, bins=25, color=ORANGE,
                 edgecolor="white", alpha=0.85)
    mean_freq = strain_freq.mean()
    axes[2].axvline(mean_freq, color="red", ls="--", lw=2,
                    label=f"Mean={mean_freq:.0f}")
    axes[2].set_title("Strain Sampling Frequency\n(should be roughly uniform = random sampling)")
    axes[2].set_xlabel("Number of communities strain appears in")
    axes[2].set_ylabel("Count")
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig5_membership_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig5_membership_v9.png")

    # ── Figure 6: Nutrient Environment Characterisation ───────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 6 — Nutrient Environments Across Diversity Levels (v9)\n"
                 "(rows grouped by diversity level; total supply constant across all)",
                 fontsize=12, fontweight="bold")

    # Left: R0 matrix — rows=broths, cols=nutrients, grouped by diversity level
    r0_matrix = np.array([b["R0"] for b in broths])
    broth_labels = [f"{b['name']}\n(D={b['diversity_level']})" for b in broths]
    im = axes[0].imshow(r0_matrix, aspect="auto", cmap="YlOrRd")
    axes[0].set_yticks(range(N_BROTHS))
    axes[0].set_yticklabels(broth_labels, fontsize=7)
    axes[0].set_xticks(range(N_RESOURCES))
    axes[0].set_xticklabels([f"c{k+1}" for k in range(N_RESOURCES)], fontsize=7, rotation=90)
    axes[0].set_title("R0 Matrix: Broth × Nutrient\n(each row is one chemical environment)")
    plt.colorbar(im, ax=axes[0], label="Initial nutrient conc. R0_α")

    # Middle: mean R0 concentration per nutrient, by diversity level
    for i, (lvl, color) in enumerate(zip(DIVERSITY_LEVELS, palette5)):
        lvl_rows = np.array([b["R0"] for b in broths if b["diversity_level"] == lvl])
        mean_r0  = lvl_rows.mean(axis=0)
        axes[1].bar(np.arange(N_RESOURCES) + i * 0.18 - 0.36,
                    mean_r0, width=0.17, alpha=0.85, color=color,
                    label=f"D={lvl}")
    axes[1].set_xticks(range(0, N_RESOURCES, 2))
    axes[1].set_xticklabels([f"c{k+1}" for k in range(0, N_RESOURCES, 2)], fontsize=8)
    axes[1].set_title("Mean R0 per Nutrient by Diversity Level\n(taller = supplied more often)")
    axes[1].set_xlabel("Nutrient")
    axes[1].set_ylabel("Mean R0 (across replicates)")
    axes[1].legend(fontsize=8)

    # Right: per-nutrient supply concentration vs diversity level
    # Shows how R0 per nutrient decreases as diversity increases (normalization)
    per_nut_conc = []
    for lvl in DIVERSITY_LEVELS:
        lvl_broths = [b for b in broths if b["diversity_level"] == lvl]
        # For each broth, compute mean concentration of ACTIVE nutrients
        active_means = []
        for b in lvl_broths:
            active_vals = b["R0"][b["R0"] > 0]
            if len(active_vals) > 0:
                active_means.append(active_vals.mean())
        per_nut_conc.append(np.mean(active_means))
    axes[2].plot(range(len(DIVERSITY_LEVELS)), per_nut_conc, color=TEAL,
                 lw=2.5, marker="o", ms=8, markerfacecolor="white",
                 markeredgewidth=2, markeredgecolor=TEAL)
    # Theoretical: TOTAL / n_active
    theo = [TOTAL_CARBON_SUPPLY / lvl for lvl in DIVERSITY_LEVELS]
    axes[2].plot(range(len(DIVERSITY_LEVELS)), theo, color=ORANGE,
                 lw=1.5, ls="--", label="Theory: 20/n_active")
    axes[2].set_xticks(range(len(DIVERSITY_LEVELS)))
    axes[2].set_xticklabels([str(l) for l in DIVERSITY_LEVELS])
    axes[2].set_title("Per-Nutrient Concentration vs Diversity Level\n"
                      "(shows normalization: total supply constant)")
    axes[2].set_xlabel("Diversity level (# active nutrients)")
    axes[2].set_ylabel("Mean R0 per active nutrient")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig6_nutrient_environments_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig6_nutrient_environments_v9.png")

    # ── Figure 7: Pairwise Suppression Heatmap ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Figure 7 — Pairwise Suppression Heatmap (60-strain sample)\n"
                 "(Green = suppresses pathogen | Yellow-Red = neutral/facilitation)",
                 fontsize=12, fontweight="bold")

    # Get ss_suppression_raw per (strain, diversity_level) from community data
    strain_broth_supp = df.groupby(["strain_id", "diversity_level"])["ss_suppression_raw"].mean().reset_index()
    sample_strains = sorted(df_mono.nlargest(30, "ss_suppression_mean")["strain_id"].tolist() +
                             df_mono.nsmallest(30, "ss_suppression_mean")["strain_id"].tolist())
    hm = strain_broth_supp[strain_broth_supp["strain_id"].isin(sample_strains)]
    hm_pivot = hm.pivot(index="strain_id", columns="diversity_level",
                        values="ss_suppression_raw").fillna(0)
    hm_pivot.columns = [f"D={c}" for c in hm_pivot.columns]

    sns.heatmap(hm_pivot, ax=axes[0], cmap="RdYlGn",
                center=0, vmin=-0.2, vmax=1.0,
                xticklabels=True, yticklabels=True,
                linewidths=0.3, linecolor="white",
                cbar_kws={"label": "ss_suppression_raw"})
    axes[0].set_title("ss_suppression by Strain × Diversity Level\n"
                      "(60 strains: top 30 + bottom 30 by mean suppression)")
    axes[0].set_xlabel("Diversity level")
    axes[0].set_ylabel("strain_id")
    axes[0].tick_params(axis="y", labelsize=6)

    # Right: ranked mean suppression
    strain_mean = df_mono[["strain_id", "species_type", "ss_suppression_mean"]].copy()
    strain_mean = strain_mean.sort_values("ss_suppression_mean", ascending=True)
    bottom_top  = pd.concat([strain_mean.head(30), strain_mean.tail(30)])
    colors_bt   = ["#4CAF50" if t == "generalist" else "#2196F3"
                   for t in bottom_top["species_type"]]
    axes[1].barh(range(len(bottom_top)), bottom_top["ss_suppression_mean"].values,
                 color=colors_bt, alpha=0.8, edgecolor="white")
    axes[1].set_yticks(range(len(bottom_top)))
    axes[1].set_yticklabels(bottom_top["strain_id"].values, fontsize=6)
    axes[1].axvline(0, color="black", lw=0.8)
    axes[1].set_title("Ranked Mean Suppression (bottom 30 + top 30)")
    axes[1].set_xlabel("Mean ss_suppression_raw (across all broths)")
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color="#2196F3", label="Specialist"),
                              Patch(color="#4CAF50", label="Generalist")],
                   fontsize=9, loc="lower right")

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig7_pairwise_suppression_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig7_pairwise_suppression_v9.png")

    # ── Figure 8: Ecological Summary ─────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle("Figure 8 — Ecological Summary: MiCRM Trait Relationships",
                 fontsize=13, fontweight="bold")
    axes = axes.flatten()

    strain_eco = df_mono.copy()
    strain_eco["n_active"] = (strain_eco[[f"uptake_c{k+1}" for k in range(N_RESOURCES)]] > 0).sum(axis=1)
    type_colors = {"specialist": "#2196F3", "generalist": "#4CAF50"}

    # Maintenance vs suppression
    for stype, col in type_colors.items():
        mask = strain_eco["species_type"] == stype
        axes[0].scatter(strain_eco.loc[mask, "maintenance_cost"],
                        strain_eco.loc[mask, "ss_suppression_mean"],
                        c=col, s=20, alpha=0.6, label=stype.capitalize())
    rho0, p0 = spearmanr(strain_eco["maintenance_cost"], strain_eco["ss_suppression_mean"])
    axes[0].set_title(f"Maintenance vs Suppression\nSpearman ρ={rho0:.3f}, p={p0:.2e}")
    axes[0].set_xlabel("Maintenance cost mi")
    axes[0].set_ylabel("Mean suppression")

    # Growth rate vs suppression
    for stype, col in type_colors.items():
        mask = strain_eco["species_type"] == stype
        axes[1].scatter(strain_eco.loc[mask, "growth_rate"],
                        strain_eco.loc[mask, "ss_suppression_mean"],
                        c=col, s=20, alpha=0.6)
    rho1, _ = spearmanr(strain_eco["growth_rate"], strain_eco["ss_suppression_mean"])
    axes[1].set_title(f"Growth Rate vs Suppression\nSpearman ρ={rho1:.3f}")
    axes[1].set_xlabel("Growth rate scalar gi")
    axes[1].set_ylabel("Mean suppression")

    # Uptake breadth vs suppression
    for stype, col in type_colors.items():
        mask = strain_eco["species_type"] == stype
        axes[2].scatter(strain_eco.loc[mask, "n_active"],
                        strain_eco.loc[mask, "ss_suppression_mean"],
                        c=col, s=20, alpha=0.6)
    rho2, _ = spearmanr(strain_eco["n_active"], strain_eco["ss_suppression_mean"])
    axes[2].set_title(f"Uptake Breadth vs Suppression\nSpearman ρ={rho2:.3f}")
    axes[2].set_xlabel("Number of active nutrients (uptake breadth)")
    axes[2].set_ylabel("Mean suppression")

    # Histogram by species type
    for stype, col in type_colors.items():
        mask = strain_eco["species_type"] == stype
        n = mask.sum()
        axes[3].hist(strain_eco.loc[mask, "ss_suppression_mean"], bins=20,
                     alpha=0.7, color=col, edgecolor="white",
                     label=f"{stype.capitalize()} (n={n})")
    axes[3].axvline(0, color="black", ls="--", lw=1.5, label="Neutral")
    axes[3].set_title("Pairwise Suppression by Species Type")
    axes[3].set_xlabel("Mean ss_suppression_raw")
    axes[3].set_ylabel("Count")
    axes[3].legend(fontsize=9)

    # Per diversity-level suppression by type
    supp_cols = [f"ss_supp_d{l}" for l in DIVERSITY_LEVELS]
    for j, (col, lvl) in enumerate(zip(supp_cols, DIVERSITY_LEVELS)):
        y_vals = [strain_eco.loc[strain_eco["species_type"] == t, col].mean()
                  for t in ["specialist", "generalist"]]
        axes[4].bar([j - 0.2, j + 0.2], y_vals, width=0.35,
                    color=["#2196F3", "#4CAF50"],
                    alpha=0.85, edgecolor="white")
    axes[4].set_xticks(range(len(DIVERSITY_LEVELS)))
    axes[4].set_xticklabels([f"D={l}" for l in DIVERSITY_LEVELS])
    axes[4].set_title("Mean Suppression per Diversity Level by Species Type")
    axes[4].set_xlabel("Diversity level")
    axes[4].set_ylabel("Mean ss_suppression_raw")
    from matplotlib.patches import Patch
    axes[4].legend(handles=[Patch(color="#2196F3", label="Specialist"),
                              Patch(color="#4CAF50", label="Generalist")],
                   fontsize=9)

    # Community suppression per broth (ordered by diversity level)
    broth_order = [b["name"] for b in sorted(broths, key=lambda x: (x["diversity_level"], x["name"]))]
    means_b = df_idx.groupby("broth_id")["community_suppression"].mean().reindex(broth_order)
    bcolors = []
    for bn in broth_order:
        lvl = next(b["diversity_level"] for b in broths if b["name"] == bn)
        bcolors.append(palette5[DIVERSITY_LEVELS.index(lvl)])
    axes[5].barh(range(len(broth_order)), means_b.values,
                 color=bcolors, alpha=0.85, edgecolor="white")
    axes[5].set_yticks(range(len(broth_order)))
    axes[5].set_yticklabels(
        [f"{bn} (D={next(b['diversity_level'] for b in broths if b['name']==bn)})"
         for bn in broth_order], fontsize=7)
    axes[5].axvline(df_idx["community_suppression"].mean(), color=ORANGE,
                    ls="--", lw=1.5, label=f"Overall mean={df_idx['community_suppression'].mean():.2f}")
    axes[5].set_title("Mean Community Suppression per Broth\n(grouped by diversity level)")
    axes[5].set_xlabel("Mean community suppression")
    axes[5].legend(fontsize=9)

    for ax in axes[:5]:
        ax.grid(True, alpha=0.25)
    for ax in axes[:3]:
        ax.legend(handles=[plt.scatter([], [], c=c, s=30, label=t.capitalize())
                            for t, c in type_colors.items()], fontsize=8)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig8_ecological_summary_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig8_ecological_summary_v9.png")

    # ── Figure 9: Individual vs Community Suppression ─────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 9 — ss_suppression_raw vs Community Suppression\n"
                 "(weak correlation = community context matters = XGBoost/GNN justified)",
                 fontsize=12, fontweight="bold")

    # Left: individual vs community, coloured by diversity level
    for lvl, col in zip(DIVERSITY_LEVELS, palette5):
        mask = df["diversity_level"] == lvl
        axes[0].scatter(df.loc[mask, "ss_suppression_raw"],
                        df.loc[mask, "community_suppression"],
                        c=col, s=3, alpha=0.2, label=f"D={lvl}", rasterized=True)
    rho_ic, p_ic = spearmanr(df["ss_suppression_raw"], df["community_suppression"])
    axes[0].set_title(f"Individual vs Community Suppression\n"
                      f"Spearman ρ={rho_ic:.3f} (p={p_ic:.2e})\n"
                      f"(weak = XGBoost with composition features needed)")
    axes[0].set_xlabel("ss_suppression_raw (individual ability)")
    axes[0].set_ylabel("Community suppression")
    axes[0].legend(fontsize=8, markerscale=4)

    # Middle: broth-level correlation
    broth_summary = df_idx.groupby(["broth_id", "diversity_level"]).agg(
        mean_comm_supp=("community_suppression", "mean")
    ).reset_index()
    mean_ind = df.groupby(["broth_id", "diversity_level"])["ss_suppression_raw"].mean().reset_index()
    bl = broth_summary.merge(mean_ind, on=["broth_id", "diversity_level"])
    for lvl, col in zip(DIVERSITY_LEVELS, palette5):
        mask = bl["diversity_level"] == lvl
        axes[1].scatter(bl.loc[mask, "ss_suppression_raw"],
                        bl.loc[mask, "mean_comm_supp"],
                        c=col, s=80, label=f"D={lvl}", zorder=3, alpha=0.85)
        for _, row in bl[mask].iterrows():
            axes[1].annotate(row["broth_id"], (row["ss_suppression_raw"], row["mean_comm_supp"]),
                              fontsize=6, ha="left", xytext=(3, 2), textcoords="offset points")
    rho_bl, _ = spearmanr(bl["ss_suppression_raw"], bl["mean_comm_supp"])
    axes[1].set_title(f"Broth-level Correlation\nSpearman ρ={rho_bl:.3f}")
    axes[1].set_xlabel("Mean ss_suppression_raw per broth")
    axes[1].set_ylabel("Mean community suppression per broth")
    axes[1].legend(fontsize=8)

    # Right: mean ss_suppression_raw within community vs community suppression
    comm_mean_ind = df.groupby(["community_id","community_size","diversity_level"])[
        "ss_suppression_raw"].mean().reset_index(name="mean_member_ss")
    comm_target = df_idx[["community_id","community_suppression"]].drop_duplicates()
    scatter_df = comm_mean_ind.merge(comm_target, on="community_id")
    for sz, col in zip(COMMUNITY_SIZES, palette5):
        mask = scatter_df["community_size"] == sz
        axes[2].scatter(scatter_df.loc[mask, "mean_member_ss"],
                        scatter_df.loc[mask, "community_suppression"],
                        c=col, s=4, alpha=0.25, label=f"N={sz}", rasterized=True)
    axes[2].set_title("Mean Member ss_suppression_raw vs\nCommunity Suppression by Size")
    axes[2].set_xlabel("Mean ss_suppression_raw within community")
    axes[2].set_ylabel("Community suppression")
    axes[2].legend(fontsize=8, markerscale=4)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig9_ss_vs_community_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig9_ss_vs_community_v9.png")

    # ── Figure 10: XGBoost Feature Structure ─────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 10 — XGBoost Feature Structure\n"
                 "(community membership + nutrient features = all that XGBoost sees)",
                 fontsize=12, fontweight="bold")

    # Left: fraction of species pairs that co-occur (membership overlap)
    species_cols = [c for c in df_idx.columns if c.startswith("Species_")]
    mem_matrix = df_idx[species_cols].values   # [n_communities, 100]
    cooccur = (mem_matrix.T @ mem_matrix)        # [100, 100] co-occurrence count
    cooccur_frac = cooccur / len(df_idx)
    np.fill_diagonal(cooccur_frac, 0)
    im = axes[0].imshow(cooccur_frac, aspect="auto", cmap="Blues", vmin=0, vmax=0.5)
    axes[0].set_title("Species Co-occurrence Matrix\n(fraction of communities where both present)")
    axes[0].set_xlabel("Species index")
    axes[0].set_ylabel("Species index")
    plt.colorbar(im, ax=axes[0], label="Co-occurrence fraction")

    # Middle: R0 feature distribution across all communities
    r0_df = df_idx[r0_cols]
    r0_nonzero = r0_df.values[r0_df.values > 0]
    axes[1].hist(r0_nonzero, bins=40, color=TEAL, edgecolor="white", alpha=0.85)
    axes[1].set_title("Distribution of Non-Zero R0 Values\n(active nutrient concentrations)")
    axes[1].set_xlabel("R0 concentration")
    axes[1].set_ylabel("Count")

    # Right: feature importance proxy — correlation of each R0 with suppression
    r0_corr = []
    for col in r0_cols:
        rho, _ = spearmanr(df_idx[col], df_idx["community_suppression"])
        r0_corr.append(rho)
    bar_colors = [TEAL if r > 0 else ORANGE for r in r0_corr]
    axes[2].bar(range(N_RESOURCES), r0_corr, color=bar_colors, edgecolor="white", alpha=0.85)
    axes[2].axhline(0, color="black", lw=0.8)
    axes[2].set_xticks(range(0, N_RESOURCES, 2))
    axes[2].set_xticklabels([f"c{k+1}" for k in range(0, N_RESOURCES, 2)], fontsize=8, rotation=45)
    axes[2].set_title("Nutrient–Suppression Correlation\n(Spearman ρ for each R0 nutrient vs suppression)")
    axes[2].set_xlabel("Nutrient")
    axes[2].set_ylabel("Spearman ρ")
    axes[2].grid(True, alpha=0.25, axis="y")

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig10_xgboost_features_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig10_xgboost_features_v9.png")

    # ── Figure 11: T_MEASURE Sensitivity Analysis ─────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Figure 11 — T_MEASURE Sensitivity Analysis\n"
                 f"(choose T where suppression has good variance, pathogen not zero)",
                 fontsize=12, fontweight="bold")

    # Run one test community across multiple T values
    broth_test = broths[N_BROTHS // 2]
    N_path_test = pathogen_solo[broth_test["name"]]
    test_comm_size = 24
    test_members = list(range(1, test_comm_size + 1))
    t_range = [1, 3, 5, 7, 10, 14, 20, 30, 50, 80, 120, 200]
    supp_t = []
    path_t = []
    for t in t_range:
        N_t, _, _ = run_micrm(
            c_matrix, broth_test["w"], broth_test["l"], m, g,
            broth_test["delta"], broth_test["R0"], D_list,
            species_subset=test_members + [PATHOGEN_IDX], t_end=t
        )
        supp_t.append(calc_suppression(N_t[PATHOGEN_IDX], N_path_test))
        path_t.append(N_t[PATHOGEN_IDX])

    axes[0].plot(t_range, supp_t, color=TEAL, lw=2, marker="o",
                 ms=6, markerfacecolor="white", markeredgewidth=2)
    axes[0].axvline(T_MEASURE, color=ORANGE, ls="--", lw=2,
                    label=f"Current T_MEASURE={T_MEASURE}")
    axes[0].axhline(0.95, color="red", ls=":", lw=1.5, label="0.95 threshold")
    axes[0].set_xscale("log")
    axes[0].set_title(f"Suppression vs Measurement Time\n"
                      f"(one test community, N={test_comm_size}, "
                      f"D={broth_test['diversity_level']})")
    axes[0].set_xlabel("Measurement time T (log scale)")
    axes[0].set_ylabel("Community suppression")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_range, path_t, color=TEAL, lw=2, marker="s",
                 ms=6, markerfacecolor="white", markeredgewidth=2)
    axes[1].axvline(T_MEASURE, color=ORANGE, ls="--", lw=2,
                    label=f"T_MEASURE={T_MEASURE}")
    axes[1].set_xscale("log")
    axes[1].set_title(f"Pathogen Solo Abundance vs Time\n(should be > 0 at chosen T_MEASURE)")
    axes[1].set_xlabel("Measurement time T (log scale)")
    axes[1].set_ylabel("N_pathogen_solo")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig11_tmeasure_sensitivity_v9.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig11_tmeasure_sensitivity_v9.png")


# =============================================================================
# 10.  DIAGNOSTIC PRINT
# =============================================================================
def print_diagnostics(df_idx):
    print("\n" + "=" * 65)
    print("  DATASET DIAGNOSTICS")
    print("=" * 65)

    print("\n  Suppression by diversity level:")
    print(df_idx.groupby("diversity_level")["community_suppression"]
          .agg(["mean", "std", "count"]).round(3).to_string())

    print("\n  Suppression by community size:")
    print(df_idx.groupby("community_size")["community_suppression"]
          .agg(["mean", "std"]).round(3).to_string())

    bw_std = df_idx.groupby("broth_id")["community_suppression"].mean().std()
    wi_std = df_idx.groupby("broth_id")["community_suppression"].std().mean()
    div_std = df_idx.groupby("diversity_level")["community_suppression"].mean().std()
    frac_hi = (df_idx["community_suppression"] > 0.95).mean()
    frac_lo = (df_idx["community_suppression"] < 0.05).mean()

    print(f"\n  Between-broth std   : {bw_std:.4f}  (target > 0.05)")
    print(f"  Between-level std   : {div_std:.4f}  (diversity effect size)")
    print(f"  Within-broth  std   : {wi_std:.4f}  (composition effect residual)")
    print(f"  Signal-to-noise     : {bw_std / (wi_std + 1e-8):.3f}  (target > 0.3)")
    print(f"  Fraction > 0.95     : {frac_hi:.3f}  (target < 0.25)")
    print(f"  Fraction < 0.05     : {frac_lo:.3f}")
    print(f"\n  Total XGBoost features per community:")
    print(f"    Binary membership  : {N_COMMUNITY} (one column per strain)")
    print(f"    R0 concentrations  : {N_RESOURCES}")
    print(f"    Active flags       : {N_RESOURCES}")
    print(f"    Fractional supply  : {N_RESOURCES}")
    print(f"    community_size     : 1")
    print(f"    diversity_level    : 1")
    print(f"    TOTAL              : {N_COMMUNITY + N_RESOURCES*3 + 2}")


# =============================================================================
# 11.  MAIN
# =============================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Generate all data
    df, genes, c_matrix, uptake_mask, m, g, broths, D_list, \
        species_type, ss_lum, pathogen_solo = generate_dataset()

    # Build community index (XGBoost-ready)
    print("\n[8] Building community index (XGBoost-ready format) ...")
    df_idx = build_community_index(df)

    # Build monoculture dataset (reference only, not XGBoost features)
    df_mono = build_monoculture_dataset(genes, c_matrix, m, g, broths,
                                         species_type, ss_lum, pathogen_solo)

    # Save
    comm_path  = os.path.join(OUTPUT_DIR, "community_dataset_v9.csv")
    index_path = os.path.join(OUTPUT_DIR, "community_index_v9.csv")
    mono_path  = os.path.join(OUTPUT_DIR, "monoculture_dataset_v9.csv")

    df.to_csv(comm_path,  index=False)
    df_idx.to_csv(index_path, index=False)
    df_mono.to_csv(mono_path,  index=False)

    print(f"\n  Saved:")
    print(f"    {comm_path}")
    print(f"    {index_path}  (XGBoost input — use this file!)")
    print(f"    {mono_path}")

    print_diagnostics(df_idx)

    # Generate figures
    print("\n[9] Generating figures ...")
    make_figures(df, df_idx, df_mono, broths, pathogen_solo, OUTPUT_DIR)
    print("\n  All figures saved.")

    print("\n  Done.")

  MiCRM v9 — Nutrient-Diversity-Aware Dataset
  T_MEASURE = 14.0
  Diversity levels: [1, 2, 4, 8, 16] (carbon sources)
  Replicates per level: {1: 10, 2: 8, 4: 6, 8: 4, 16: 4}  →  32 broths total
  Community sizes: [12, 16, 24, 32, 48]
  Samples per size per broth: 300
  Expected communities: 48,000
  Total carbon supply (constant): 20.0

[1] Building species traits ...
[2] Building broth environments (diversity-structured, v8) ...
   Broth_01    diversity= 1  active=['c9']  R0_sum=20.00
   Broth_02    diversity= 1  active=['c9']  R0_sum=20.00
   Broth_03    diversity= 1  active=['c1']  R0_sum=20.00
   Broth_04    diversity= 1  active=['c12']  R0_sum=20.00
   Broth_05    diversity= 1  active=['c8']  R0_sum=20.00
   Broth_06    diversity= 1  active=['c13']  R0_sum=20.00
   Broth_07    diversity= 1  active=['c4']  R0_sum=20.00
   Broth_08    diversity= 1  active=['c16']  R0_sum=20.00
   Broth_09    diversity= 1  active=['c11']  R0_sum=20.00
   Broth_10    diversity= 1  active=['c17']  R0

  Pathogen solo: 100%|██████████████████████████████████████████████████████████████| 32/32 [00:00<00:00, 37.40broth/s]



[5] Single-strain suppression (ss_lum, 100 strains × 32 broths) ...


  Pairwise ODE: 100%|███████████████████████████████████████████████████████████████| 32/32 [02:37<00:00,  4.91s/broth]



[6] Community suppression (48,000 communities) ...


  Community ODE: 100%|███████████████████████████████████████████████████████████| 32/32 [8:17:25<00:00, 932.66s/broth]



  Total rows (strain × community): 1,267,200
  Unique communities             : 48,000

  community_suppression summary:
count    1.267200e+06
mean     7.006000e-01
std      3.754000e-01
min      0.000000e+00
25%      5.586000e-01
50%      8.971000e-01
75%      9.784000e-01
max      9.990000e-01
Name: community_suppression, dtype: float64

  By diversity level:
                  mean    std   count
diversity_level                      
1                0.524  0.454  396000
2                0.621  0.409  316800
4                0.923  0.128  237600
8                0.827  0.176  158400
16               0.843  0.148  158400

[8] Building community index (XGBoost-ready format) ...
  Community index: 48,000 rows × 168 cols
  XGBoost feature count: 142 (R0×20 + active×20 + membership×100 + size + level)

[7] Building monoculture dataset ...
  Monoculture dataset: 100 rows × 59 cols


OSError: Cannot save file into a non-existent directory: 'C:\Users\Pratham Shah\Downloads\Tabular_Models_Dataset'

In [21]:
# 1. PASTE YOUR EXACT FOLDER PATH INSIDE THE QUOTES BELOW:
# (Keep the 'r' outside the quotes to handle Windows backslashes properly)
TARGET_DIR = r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset"


# ==========================================
# NO FORCE-CREATION. JUST SAVING YOUR DATA.
# ==========================================
import os

comm_path = os.path.join(TARGET_DIR, "community_dataset_v9.csv")
index_path = os.path.join(TARGET_DIR, "community_index_v9.csv")
mono_path  = os.path.join(TARGET_DIR, "monoculture_dataset_v9.csv")

print("Attempting to save variables to your specified folder...")

try:
    # Save the dataframes from RAM
    df.to_csv(comm_path, index=False)
    df_idx.to_csv(index_path, index=False)
    df_mono.to_csv(mono_path, index=False)
    print("\n✅ SUCCESS! Data saved to your folder:")
    print(f"   - {comm_path}")
    print(f"   - {index_path}")
    print(f"   - {mono_path}")
    
    # Run the remaining console diagnostics
    print("\n[8.1] Running dataset diagnostics...")
    print_diagnostics(df_idx)
    
    # Generate the figures and save them to your folder
    print("\n[9] Generating and saving figures...")
    make_figures(df, df_idx, df_mono, broths, pathogen_solo, TARGET_DIR)
    print("✅ All figures generated and saved successfully!")

except Exception as e:
    print(f"\n❌ Error saving to that directory: {e}")
    print("Your data is still safe in RAM. Double check the path formatting and try again.")

Attempting to save variables to your specified folder...

✅ SUCCESS! Data saved to your folder:
   - C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\community_dataset_v9.csv
   - C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\community_index_v9.csv
   - C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\monoculture_dataset_v9.csv

[8.1] Running dataset diagnostics...

  DATASET DIAGNOSTICS

  Suppression by diversity level:
                  mean    std  count
diversity_level                     
1                0.504  0.449  15000
2                0.596  0.410  12000
4                0.891  0.163   9000
8                0.781  0.208   6000
16               0.790  0.176   6000

  Suppression by community size:
                 mean    std
community_size              
12              0.565  0.372
16              0.620  0.370
24              0.689  0.368
32              0.721  0.369
48              0.753  0.373

  Between-broth std   : 0.345